In [ ]:
%load_ext autoreload
%autoreload 2

# ignore warnings for readability
import warnings
warnings.filterwarnings('ignore')

import os
from os.path import join
import numpy as np
import sklearn.neighbors as skn
from sklearn.linear_model import LinearRegression
import tqdm
from scipy.stats import norm
import json
import pandas as pd
from collections import defaultdict
import pandas as pd
import logging
logging.getLogger('fontTools.subset').setLevel(logging.WARNING)


# torch
import torch
import torch_geometric as pyg

# matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
# mpl.style.use('style.mcstyle')
mpl.style.use('euclid_stylesheet_v2.mplstyle')
mpl.rcParams['figure.dpi'] = 300
import seaborn as sns

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

# ili
import ili
from ili.dataloaders import NumpyLoader, StaticNumpyLoader
from ili.inference.runner_sbi import SBIRunner
from ili.validation.metrics import PlotSinglePosterior, PosteriorCoverage
from ili.utils import IndependentTruncatedNormal, Uniform, IndependentNormal

from tools.plot_tools import binned_plot

In [ ]:
mdir = './saved_models'


def r2logm(r):
    # see preprocessing.ipynb for this measurement
    coef = 0.36752
    intercept = -5.30640
    return (np.log10(r)-intercept)/coef

In [ ]:
datanames = ['wC50', 'wC100', 'dC50', 'dC100']
runnames = ['base', 'gals_nle', 'summ_nle', 'gnn_npe']
modelnames = ['msig', 'pamico', 'mamp', 'gals_nle', 'summ_nle', 'gnn_npe']
Nfolds = 10
folds = np.arange(Nfolds)

datatitles = dict(
    wC50='Wide $50\%$',
    wC100='Wide $100\%$',
    dC50='Deep $50\%$',
    dC100='Deep $100\%$'
)
modeltitles = dict(
    msig='$M$--$\sigma$',
    pamico=r'$M$--$\lambda_{\rm spec}$',
    mamp='MAMPOSSt',
    gals_nle='Galaxy-Net',
    summ_nle='Summary-Net',
    gnn_npe='Graph-Net',
    true='True',
    prior='Prior'
)
latextitles = dict(
    msig='\Msig',
    pamico=r'$M$--$\lambda_{\rm spec}$',
    mamp='MAMPOSSt',
    gals_nle='Galaxy-Net',
    summ_nle='Summary-Net',
    gnn_npe='Graph-Net',
    true='True',
    prior='Prior'
)

In [ ]:
from scipy.stats import skewnorm
from scipy.optimize import minimize


def fit_skewed_normal(p16, p50, p84):
    target_percentiles = [0.16, 0.50, 0.84]
    observed_values = [p16, p50, p84]

    def objective(params):
        loc, scale, alpha = params
        if scale <= 0:
            return np.inf
        skewed_gaussian = skewnorm(alpha, loc=loc, scale=scale)
        calculated_values = skewed_gaussian.ppf(target_percentiles)
        return np.sum((calculated_values - observed_values) ** 2)

    initial_guess = [p50, (p84 - p16) / 2, 0.5]
    result = minimize(objective, initial_guess)
    loc, scale, alpha = result.x
    return skewnorm(alpha, loc=loc, scale=scale)

In [ ]:
# load train/test
header = 'APR24'
datadir = './data/processed'

theta, fold, ids, richs, zclus, srichs, Ngal = {}, {}, {}, {}, {}, {}, {}
for d in datanames:
    dirpath = join(datadir, f'{header}{d}')
    print('Loading:', dirpath)
    theta[d] = np.load(join(dirpath, 'theta_batch.npy'))
    fold[d] = np.load(join(dirpath, 'folds_batch.npy'))
    ids[d] = np.load(join(dirpath, 'ids_batch.npy'))
    # _s = np.load(join(dirpath, 'x_sum.npy'))
    # richs[d] = _s[:,3]  # old richnesses (sum of AMICO+spectra)
    metas = np.load(join(dirpath, 'metas_batch.npy'))
    zclus[d] = metas[:, 3]  # cluster photometric redshift
    richs[d] = metas[:, 1]  # sum of AMICO photometry
    srichs[d] = metas[:, 5]  # sum of spectra photometry
    Ngal[d] = np.load(join(dirpath, 'x_sum.npy'))[:, -1]

Ndata = {d: len(theta[d]) for d in datanames}
print(Ndata)

In [ ]:
Nsamp = 100
preds = defaultdict(dict)
for d in datanames:
    for r in runnames:
        # setup
        if r == 'base':
            preds[d]['msig'] = np.ones((Ndata[d], 2))*np.nan
            preds[d]['pamico'] = np.ones((Ndata[d], 2))*np.nan
        else:
            preds[d][r] = np.empty((Ndata[d], Nsamp, 1))

        # load
        for f in folds:
            if r == 'gnn_npe':
                dirname = f'oct02_{r}_{d}_f{f}'
            else:
                dirname = f'apr24_{r}_{d}_f{f}'
            # dirname = f'apr24_{r}_{d}_f{f}'
            if r == 'base':
                # Msig
                samplefile = join(mdir, dirname, 'msig.npz')
                if not os.path.exists(samplefile):
                    print(f'Skipping {dirname}')
                    continue
                s = np.load(samplefile)
                place_ids = np.searchsorted(ids[d], s['ids'])
                np.put(preds[d]['msig'][:, 0], place_ids, s['pred'])
                np.put(preds[d]['msig'][:, 1], place_ids, s['std'])

                # Pamico
                samplefile = join(mdir, dirname, 'Pamico.npz')
                if not os.path.exists(samplefile):
                    print(f'Skipping {dirname}')
                    continue
                s = np.load(samplefile)
                place_ids = np.searchsorted(ids[d], s['ids'])
                np.put(preds[d]['pamico'][:, 0], place_ids, s['pred'])
                np.put(preds[d]['pamico'][:, 1], place_ids, s['std'])

            else:
                # ML models
                samplefile = join(mdir, dirname, 'posterior_samples.npy')
                if not os.path.exists(samplefile):
                    print(f'Skipping {dirname}')
                    continue
                s = np.load(samplefile)
                s = np.swapaxes(s, 0, 1)
                s = s[:, :Nsamp]  # subsample if necessary
                preds[d][r][fold[d] == f] = s

In [ ]:
# load mamposst

mamnames = {
    'wC50': 'wide50', 'wC100': 'wide100', 'dC50': 'deep50', 'dC100': 'deep100'
}
modeldir = './saved_models/mamposst_newprior_dec1824/'
# modeldir = './saved_models/mamposst_dec1224/'

for k, v in mamnames.items():
    isamp = pd.read_csv(join(modeldir, f'result_MockFS_NewAMICO_{v}.dat'),
                        delimiter=' ', skipinitialspace=True)
    isamp['id'] = isamp['#ClusterID'].astype(int)
    # convert r200 to logm
    for c in isamp.columns:
        if 'r200' not in c:
            continue
        isamp['logm'+c[4:]] = r2logm(isamp[c])

    # put in preds
    preds[k]['mamp'] = np.ones((Ndata[k], 5))*np.nan
    place_ids = np.searchsorted(ids[k], isamp['id'].values)
    mask = place_ids < Ndata[k]
    _s = isamp[['logmlow(68)', 'logmup(68)', 'logmlow(95)',
                'logmup(95)', 'logmMAM']].values
    preds[k]['mamp'][place_ids[mask]] = _s[mask]

In [ ]:
# calculate percentiles from predictions
q = 100*np.array([0.16, 0.84, 0.5, 0.025, 0.975])
percs = defaultdict(dict)
for d in datanames:
    for m in modelnames:
        if m == 'msig' or m == 'pamico':
            t_ = preds[d][m]
            percs[d][m] = np.stack(
                [t_[:, 0]-t_[:, 1], t_[:, 0]+t_[:, 1], t_[:, 0],
                    t_[:, 0]-2*t_[:, 1], t_[:, 0]+2*t_[:, 1]],
                axis=1).T
        elif m == 'mamp':
            if m not in preds[d]:
                continue
            t_ = preds[d][m]
            percs[d][m] = np.stack(
                [t_[:, 0], t_[:, 1], t_[:, 4], t_[:, 2], t_[:, 3]],
                axis=1).T
        else:
            t_ = preds[d][m]
            percs[d][m] = np.percentile(t_, q, axis=1)[..., 0]
# percs is of shape (5, Ndata)
# dim 0 is of order [16, 84, 50, 2.5, 97.5]

In [ ]:
# Compute quality control
def quality_control(percs):
    # checks if we have a reasonable median prediction
    # checks if we're not missing a prediction (not nan)
    med = percs[2]
    mask = (med > 12) & (med < 16)

    err = (percs[1] - percs[0])/2
    mask &= err < 1
    return mask


qc = defaultdict(dict)
for d in datanames:
    for m in modelnames:
        qc[d][m] = quality_control(percs[d][m])

In [ ]:
# Remove Ngal < 3
for d in datanames:
    mask = Ngal[d] >= 3

    theta[d] = theta[d][mask]
    fold[d] = fold[d][mask]
    ids[d] = ids[d][mask]
    richs[d] = richs[d][mask]
    zclus[d] = zclus[d][mask]
    srichs[d] = srichs[d][mask]
    Ngal[d] = Ngal[d][mask]
    Ndata[d] = len(theta[d])

    for m in modelnames:
        if m not in percs[d]:
            continue
        preds[d][m] = preds[d][m][mask]
        percs[d][m] = percs[d][m][:, mask]
        qc[d][m] = qc[d][m][mask]

## MCMC

In [ ]:

# LinearRegression to fit prior sigma_lambda
d = 'dC100'
rs = richs[d]
zs = zclus[d]
ytrue = theta[d][:, 0]

# # quality control
# mask = qc[d][m]
# rs, zs, samps, ytrue = rs[mask], zs[mask], samps[mask], ytrue[mask]

# # richness cut
# mask = rs > 0
# rs, zs, samps, ytrue = rs[mask], zs[mask], samps[mask], ytrue[mask]

# find pivot
l0, z0 = 15, 1.0  # rs.mean(), zs.mean()

# calculate sigma_lambda
rs = np.log10(rs/l0)
zs = np.log10((1+zs)/(1+z0))
X = np.stack([rs, zs], axis=1)
reg = LinearRegression().fit(X, ytrue)
y_pred = reg.predict(X)
residual_std = np.std(ytrue - y_pred)
print(f'sigma_lambda: {residual_std:.3f}')

In [ ]:
import jax
import jax.numpy as jnp
from jax.scipy.special import logsumexp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
import numpy as np
import corner
import matplotlib.pyplot as plt


def mlamb_mean(lambs, zs, l0, z0, A, B, C):
    return A + B*(jnp.log10(lambs/l0)) + C*(jnp.log10((1+zs)/(1+z0)))


def operand(m, lambs, zs, l0, z0, m0, sig0, sigl, A, B, C):
    t1 = (m-m0)**2/(2*(sig0**2))
    mest = mlamb_mean(lambs, zs, l0, z0, A, B, C)
    t2 = (m - mest[:, None])**2/(2*(sigl**2))
    return t1-t2


# def log_likelihood(samps, lambs, zs, l0, z0, m0, sig0, sigl, A, B, C):
#     Nclu, Nsamp = samps.shape
#     to_sum = operand(samps, lambs, zs, l0, z0, m0, sig0, sigl, A, B, C)
#     summed = logsumexp(to_sum, axis=1)
#     out = jnp.log(sigl) - jnp.log(Nsamp) + jnp.mean(summed)
#     return out

def log_likelihood(samps, lambs, zs, l0, z0, m0, sig0, sigl, A, B, C):
    Nclu, Nsamp = samps.shape
    to_sum = operand(samps, lambs, zs, l0, z0, m0, sig0, sigl, A, B, C)
    # LSE over samples, (Nclu,)
    summed = logsumexp(to_sum, axis=1)
    out = -Nclu*jnp.log(sigl) - Nclu*jnp.log(Nsamp) + jnp.sum(summed)
    return out

# %timeit log_likelihood(mest, lis, zis, m0, sig0, sigl, A, B, C)


def model(samps, lambs, zs, l0, z0, m0, sig0):
    A = numpyro.sample("A", dist.Uniform(5, 20))
    B = numpyro.sample("B", dist.Uniform(-10, 10))
    C = numpyro.sample("C", dist.Uniform(-10, 10))
    # sigl = numpyro.sample("sigl", dist.LogNormal(-1.407,0.136))
    # sigl = numpyro.sample("sigl", dist.Uniform(0., 0.6))
    sigl = numpyro.sample("sigl", dist.LogUniform(1e-1, 0.6))
    # sigl = numpyro.sample("sigl", dist.Exponential(4.))
    # sigl = numpyro.sample("sigl", dist.InverseGamma(8.864, 2.060))

    numpyro.factor(
        "log_likelihood",
        log_likelihood(samps, lambs, zs, l0, z0, m0, sig0, sigl, A, B, C))


def run_mcmc(samps, lambs, zs, l0, z0, m0, sig0,
             warmup=500, samples=1000, chains=4):
    mcmc = MCMC(NUTS(model), num_warmup=warmup,
                num_samples=samples, num_chains=chains)
    mcmc.run(jax.random.PRNGKey(0), samps=samps, lambs=lambs,
             zs=zs, l0=l0, z0=z0, m0=m0, sig0=sig0)
    return mcmc.get_samples()

In [ ]:

d = 'wC100'
# d = 'wC50'
# d = 'dC100'
m = 'gnn_npe'
# m = 'mamp'
rs = richs[d]
zs = zclus[d]
ytrue = theta[d][:, 0]
if m in ['msig', 'pamico']:
    mu, sig = preds[d][m].T
    samps = mu[:, None] + sig[:, None]*np.random.randn(len(mu), Nsamp)
elif m == 'mamp':
    ps = preds[d][m]
    p16, p84, p50 = ps[:, 0], ps[:, 1], ps[:, 4]
    samps = []
    for i in tqdm.tqdm(range(len(p16))):
        if np.isnan(p16[i]):
            samps.append(np.nan*np.ones(Nsamp))
            continue
        dist = fit_skewed_normal(p16[i], p50[i], p84[i])
        samps.append(dist.rvs(Nsamp))
    samps = np.array(samps)
elif m != 'true':
    samps = preds[d][m][..., 0]
else:
    samps = ytrue[:, None]
    m = 'gnn_npe'

# quality control
mask = qc[d][m]
rs, zs, samps, ytrue = rs[mask], zs[mask], samps[mask], ytrue[mask]

# # richness cut
# mask = rs > 0
# rs, zs, samps, ytrue = rs[mask], zs[mask], samps[mask], ytrue[mask]

# find pivot
l0, z0 = 15, 1.0  # rs.mean(), zs.mean()

# write down prior
m0, sig0 = 13.78, 0.35


# # TEMP (TODO: remove)
# samps = np.repeat(ytrue[:, None], repeats=100, axis=1)
# samps += np.random.randn(len(samps), 1) * 0.3
# samps += np.random.randn(*samps.shape) * 0.3

print(samps.shape)

In [ ]:
# Run MCMC
samples = run_mcmc(samps, rs, zs, l0, z0, m0, sig0, samples=2000, chains=2)
# samples = run_mcmc(ytemp, rs, zs, l0, z0, m0, sig0, samples=2000, chains=2)

# # # Load samples from file
# samples = np.load(
#     f'./saved_samples/{d}_{m}_mcmc_samples.npy', allow_pickle=True).item()
# loglik = np.load(f'./saved_samples/{d}_{m}_mcmc_loglik.npy', allow_pickle=True)

In [ ]:
noise = 0.  # *np.random.randn(1,100)
truesamples = run_mcmc(ytrue[:, None], rs, zs, l0,
                       z0, m0, sig0, samples=2000, chains=2)

# # # # Load samples from file
# truesamples = np.load(
#     f'./saved_samples/{d}_true_mcmc_samples.npy', allow_pickle=True).item()
# trueloglik = np.load(
#     f'./saved_samples/{d}_true_mcmc_loglik.npy', allow_pickle=True)

In [ ]:
# plot the corner plot
fig = plt.figure(figsize=(4.5, 4.5))

# Set parameter ranges
ranges = [
    (5, 20),
    (-10, 10),
    (-10, 10),
    (0, 0.6)
]
ranges = None

# Fit with Posterior Masses
data = np.vstack(
    [samples['A'], samples['B'], samples['C'], samples['sigl']]
).T
corner.corner(
    data,
    labels=[r"$m_0$", r"$F_\lambda$", r"$G_z$", r"$\sigma_{\rm int}$"],
    # quantiles=[0.16, 0.5, 0.84],
    levels=[0.68, 0.95],
    range=ranges,
    fig=fig,
    color='C3',
    alpha=0.1, plot_contours=True, smooth=0.5, smooth1d=0.5, hist_bin_factor=1.5,
    plot_datapoints=False, no_fill_contours=True
)

# Fit with True Masses
data = np.vstack(
    [truesamples['A'], truesamples['B'], truesamples['C'], truesamples['sigl']]
).T
corner.corner(
    data,
    labels=[r"$m_0$", r"$F_\lambda$", r"$G_z$", r"$\sigma_{\rm int}$"],
    # quantiles=[0.16, 0.5, 0.84],
    levels=[0.68, 0.95],
    range=ranges,
    fig=fig,
    color='C0',
    plot_contours=True, smooth=0.5, smooth1d=0.5, hist_bin_factor=1.5,
    plot_datapoints=False, no_fill_contours=True,
)
# Set yticks for the bottom-left subplot
# Get the bottom-left subplot and set the yticks manually
for ax in fig.axes[-4:-1]:
    ax.set_yticks([0, 0.3, 0.6])
fig.axes[-1].set_xticks([0, 0.3, 0.6])

plt.plot([], [], color='C0', label='Fit with True Masses')
plt.plot([], [], color='C3', label='Fit with Posterior Masses')

# Add the legend
plt.legend(loc='upper right', bbox_to_anchor=(1.1, 2.6), fontsize='small')

# fig.savefig(f'figures/mlambda_corner_{d}.pdf', bbox_inches='tight')

In [ ]:
f, ax = plt.subplots(1, 1, figsize=(4.5, 4.5))

ax.semilogx()

# subsample scatter plot
np.random.seed(24)
mask = np.random.choice(len(rs), 70, replace=False)

# plot samps
p_ = np.percentile(samps, [16, 50, 84], axis=1)
p_ = p_[:, mask]
ax.errorbar(rs[mask], p_[1], yerr=[p_[1]-p_[0], p_[2]-p_[1]],
            fmt='.', label='Posterior', alpha=1, elinewidth=0.75, color='C3')

# plot true
ax.plot(rs[mask], ytrue[mask],
        'x', label='True', alpha=1, markersize=5, color='C0')

# Set up fit grid
x = 10**np.linspace(np.log10(2), np.log10(110), 100)
z = z0


# Plot fit with posterior masses
s = samples
lines = mlamb_mean(x[:, None], z, l0, z0, s['A'], s['B'], s['C'])
lines += s['sigl']*np.random.randn(*lines.shape)
p_ = np.percentile(lines, [16, 50, 84], axis=1)
ax.plot(x, p_[1], label='Fit with Posterior Masses', c='C3')
ax.fill_between(x, p_[0], p_[2], alpha=0.2, color='C3')


# Plot fit with true masses
s = truesamples
lines = mlamb_mean(x[:, None], z, l0, z0, s['A'], s['B'], s['C'])
lines += s['sigl']*np.random.randn(*lines.shape)
p_ = np.percentile(lines, [16, 50, 84], axis=1)
ax.plot(x, p_[1], label='Fit with True Masses', c='C0')
ax.fill_between(x, p_[0], p_[2], alpha=0.2, color='C0')

ax.set_xlim(3, 80)
ax.set_ylim(12.5, 15.)
ax.set_xlabel(r'Photometric richness $\lambda_{\rm phot}$')
ax.set_ylabel(r'$\log_{10}\left[M_{\rm 200c}\ /\ (h^{-1}M_{\odot})\right]$')
ax.set_title(f'$\it{{Euclid}}$ {datatitles[d]} + {modeltitles[m]}')
ax.set_xticks([5, 10, 20, 50])
ax.get_xaxis().set_major_formatter(mpl.ticker.ScalarFormatter())

handles, labels = plt.gca().get_legend_handles_labels()
order = [0, 3, 2, 1]
ax.legend([handles[idx] for idx in order], [labels[idx]
                                            for idx in order], loc='lower left')


# f.savefig(f'figures/mlambda_fit_{d}.pdf', bbox_inches='tight')